# DMSTL: previsão em painel

Esta versão segue o fluxo de decomposição local por SKU, consolidação em painéis por tendência e sazonalidade e uso de um único modelo residual para todos os SKUs.

O objetivo é comparar o custo da estratégia local (um modelo residual por SKU) com a estratégia em painel (um único residual global + agregação por série), mantendo o mesmo esquema de saída.


In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / 'pyproject.toml').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Nao foi possivel localizar o diretorio do projeto.')
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal
from sklearn.linear_model import LinearRegression
from statsforecast.models import AutoETS, SeasonalNaive
from mlforecast import MLForecast

from tinyshift.modelling import DMSTLWrapper

/home/heylucasleao/forecasting/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from functools import partial

In [3]:
def make_panel(n_series=80, history=84, seed=42):
    rng = np.random.default_rng(seed)
    steps = np.arange(history)
    dates = pd.date_range("2020-01-01", periods=history, freq="D")
    frames = []

    for index in range(n_series):
        values = (
            40
            + index * 0.2
            + 0.08 * steps
            + 6 * np.sin(2 * np.pi * steps / 7 + index / 8)
            + rng.normal(scale=0.8, size=history)
        )
        frames.append(
            pd.DataFrame(
                {"unique_id": f"series-{index}", "ds": dates, "y": values}
            )
        )

    return pd.concat(frames, ignore_index=True)


def residual_model_callable(nlags, freq):
    return MLForecast(
        models=[LinearRegression()],
        lags=nlags,
        freq=freq,
    )


def seasonal_model_callable(period):
    return SeasonalNaive(season_length=period)


def trend_model_callable():
    return AutoETS(model="ZZN")


def build_model(model_class, mode):
    return model_class(
        residual_model_callable=residual_model_callable,
        freq="D",
        season_length=[7],
        trend_model_callable=trend_model_callable,
        seasonal_model_callable=seasonal_model_callable,
        nlags=[1, 2, 7],
        mode=mode,
    )


panel = make_panel()
horizon = 14
traditional = build_model(DMSTLWrapper, "local").fit(panel)
panel_model = build_model(DMSTLWrapper, "global").fit(panel)

traditional_predictions = traditional.predict(h=horizon)
panel_predictions = panel_model.predict(h=horizon)

assert len(panel_predictions) == panel["unique_id"].nunique() * horizon
assert set(panel_predictions["unique_id"]) == set(panel["unique_id"])
assert set(traditional_predictions.columns) == set(panel_predictions.columns)
print("O modelo de painel produziu previsões para todos os SKUs com o mesmo esquema de saída.")


O modelo de painel produziu previsões para todos os SKUs com o mesmo esquema de saída.


In [4]:
traditional_predictions = traditional.predict(h=horizon)
panel_predictions = panel_model.predict(h=horizon)

assert len(panel_predictions) == panel["unique_id"].nunique() * horizon
assert set(panel_predictions["unique_id"]) == set(panel["unique_id"])
assert set(traditional_predictions.columns) == set(panel_predictions.columns)
print("Validação final do painel: previsões consistentes com o esquema do modelo local.")

Validação final do painel: previsões consistentes com o esquema do modelo local.


In [ ]:
from timeit import repeat


def benchmark(predict_callable, repeats=5, calls_per_repeat=3):
    timings = repeat(predict_callable, repeat=repeats, number=calls_per_repeat)
    return np.asarray(timings) / calls_per_repeat


traditional_times = benchmark(lambda: traditional.predict(h=horizon))
panel_times = benchmark(lambda: panel_model.predict(h=horizon))

traditional_median = np.median(traditional_times)
panel_median = np.median(panel_times)
gain_pct = 100 * (traditional_median - panel_median) / traditional_median

benchmark_result = pd.DataFrame(
    {
        "implementation": ["DMSTLWrapper (local)", "PanelDMSTLWrapper (global)"],
        "median_seconds": [traditional_median, panel_median],
        "mean_seconds": [traditional_times.mean(), panel_times.mean()],
        "gain_vs_traditional_pct": [0.0, gain_pct],
    }
)
benchmark_result

,implementation,median_seconds,mean_seconds,gain_vs_traditional_pct
0,DMSTLWrapper (local),1.956366,1.968781,0.000000
1,PanelDMSTLWrapper (fluxo.txt),0.168491,0.170108,91.387563


## Leitura do resultado

Um ganho positivo indica que o batch de painel reduziu o tempo de previsao. O ganho vem de reduzir de `n_series` modelos residuais para um unico `MLForecast`, e de consolidar as chamadas de tendencia e sazonalidade em poucos objetos `StatsForecast` com `n_jobs=-1`.

Essa arquitetura e apropriada quando todos os SKUs compartilham `freq`, `nlags`, periodos sazonais e classes de modelos. Se os SKUs exigirem configuracoes distintas, agrupe-os por configuracao e crie um painel por grupo. A decomposicao MSTL continua univariada no `statsmodels`, logo o loop de decomposicao permanece no treino.